# Phase 1 — Data pipeline (USD/JPY vs US–Japan yield spread)

Runs the Phase 1 pipeline end to end: fetch FRED (UST), MoF (JGB) and Yahoo (USD/JPY), align onto one
daily snapshot calendar, write `data/processed/daily.parquet` + `metadata.json`, validate, and look at the result.

Works in **Google Colab** (clones the repo) and in **VS Code** (opened from inside the repo). Nothing in
production depends on this notebook; it only calls the package.

**Acceptance for Phase 1**: fetch + validate run clean from an empty `data/` directory and cover 2010 → present.

In [ ]:
# --- 0. Setup: locate/clone the repo, install dependencies, set the project root ---------------
import importlib, os, sys, subprocess, pathlib

REPO_URL = "https://github.com/andreapagani2003-beep/FOREX-MODEL.git"
BRANCH = "phase-2-stats"  # branch to clone; set to "main" once PR 2 is merged
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    root = pathlib.Path("/content/FOREX-MODEL")
    if not root.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(root)], check=True)
    else:
        subprocess.run(["git", "-C", str(root), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(root), "checkout", "-q", BRANCH], check=True)
        subprocess.run(["git", "-C", str(root), "reset", "-q", "--hard", f"origin/{BRANCH}"], check=True)
    # Non-editable install so the package lands in site-packages of *this* kernel. An editable
    # install (-e) only registers a .pth file, which a running kernel does not read until restart.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(root)], check=True)
else:
    # VS Code / local: walk up from the notebook's cwd until we find pyproject.toml
    here = pathlib.Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), None)
    assert root is not None, "open this notebook from inside the repo (or set root manually)"

# Belt and braces: import straight from the source tree regardless of how/if it was installed.
src = str(root / "src")
if src not in sys.path:
    sys.path.insert(0, src)
importlib.invalidate_caches()

os.environ["USDJPY_MR_ROOT"] = str(root)
os.chdir(root)
import usdjpy_mr  # noqa: E402
print("project root:", root, "| Colab:", IN_COLAB, "| usdjpy_mr", usdjpy_mr.__version__, "from", usdjpy_mr.__file__)

In [ ]:
# --- 1. Secrets: FRED API key (optional; keyless CSV fallback is used if absent) ---------------
import os
if IN_COLAB:
    try:
        from google.colab import userdata  # add FRED_API_KEY under the key icon in the left sidebar
        key = userdata.get("FRED_API_KEY")
        if key:
            os.environ["FRED_API_KEY"] = key
    except Exception as exc:  # secret not set / access not granted
        print("no Colab secret FRED_API_KEY:", type(exc).__name__)
else:
    from dotenv import load_dotenv
    load_dotenv(root / ".env")
print("FRED key present:", bool(os.environ.get("FRED_API_KEY")))

In [ ]:
# --- 2. Load config -----------------------------------------------------------------------------
from usdjpy_mr.config import load_config
cfg = load_config()  # configs/default.yaml; pass "configs/tokyo_close.yaml" to compare conventions
print("convention:", cfg.alignment.convention)
print("start:", cfg.project.start_date, "end:", cfg.project.effective_end_date())
print("processed ->", cfg.daily_parquet)

In [ ]:
# --- 3. Fetch raw sources and build daily.parquet (same code path as scripts/fetch.py) ----------
import logging
logging.getLogger("usdjpy_mr").setLevel(logging.INFO)
from usdjpy_mr.data.pipeline import fetch_all, build_daily

sources, raw_files = fetch_all(cfg)
df, meta = build_daily(cfg, sources, raw_files)
print(f"rows={len(df)}  {meta['first_date']} -> {meta['last_date']}  dropped={meta['alignment']['rows_dropped']}")
df.tail()

In [ ]:
# --- 4. Validate (same code path as scripts/validate.py) ---------------------------------------
from IPython.display import Markdown, display
from usdjpy_mr.data.validate import validate_daily

res = validate_daily(df, cfg, meta)
display(Markdown(res.to_markdown()))
assert res.ok, "validation FAILED - see errors above; do not proceed to Phase 2"

In [ ]:
# --- 5. Alignment report: what was dropped, carried forward, or unexplained --------------------
import pandas as pd
al = meta["alignment"]
print("convention:", al["convention"])
print(al["convention_note"])
print("calendar weekdays:", al["calendar_days"], "| rows out:", al["rows_out"], "| dates dropped:", al["rows_dropped"])
print("dropped by column/reason:", al["dropped_by_reason"])
print("carried-forward rows (lag>0):", al["carried_forward_rows"])
print("max lag days:", al["max_lag_days"])
gaps = pd.DataFrame({k: {"holiday": v["holiday"], "publication_lag": v.get("publication_lag", 0), "unexplained": v["unexplained"]} for k, v in al["source_gaps"].items()}).T
display(gaps)
for k, v in al["source_gaps"].items():
    if v["unexplained_dates"]:
        print(k, "unexplained gap dates (first 15):", v["unexplained_dates"][:15])

In [ ]:
# --- 6. Plots ----------------------------------------------------------------------------------
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
ax = axes[0]; ax.plot(df.index, df["usdjpy"], lw=0.8, label="USD/JPY"); ax.set_ylabel("USD/JPY"); ax.legend(loc="upper left")
ax2 = ax.twinx(); ax2.plot(df.index, df["spread10y"], lw=0.8, color="tab:orange", label="10y spread (US−JGB, %)"); ax2.set_ylabel("spread 10y"); ax2.legend(loc="lower right")
ax = axes[1]; ax.plot(df.index, df["us10y"], lw=0.8, label="UST 10y"); ax.plot(df.index, df["jgb10y"], lw=0.8, label="JGB 10y")
if "us10y_real" in df: ax.plot(df.index, df["us10y_real"], lw=0.8, label="UST 10y real (TIPS)")
ax.set_ylabel("%"); ax.legend(loc="upper left")
ax = axes[2]; ax.plot(df.index, df["us2y"], lw=0.8, label="UST 2y"); ax.plot(df.index, df["jgb2y"], lw=0.8, label="JGB 2y"); ax.set_ylabel("%"); ax.legend(loc="upper left")
for a in axes: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(12, 2.5))
for c in ("fx_lag_days", "us_lag_days", "jgb_lag_days"):
    ax.plot(df.index, df[c], lw=0.6, label=c)
ax.set_ylabel("lag (days)"); ax.legend(loc="upper left"); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# --- 7. Sanity anchors (preview for Phase 2): Jan 2023 and Jul 2024 -----------------------------
# Handoff: Jan 2023 USD/JPY ~128 with 10y spread ~3.1%; Jul 2024 USD/JPY ~161 with spread ~3.4%.
for label, sl in (("Jan 2023", slice("2023-01-10", "2023-01-20")), ("Jul 2024", slice("2024-07-05", "2024-07-12"))):
    print(label); display(df.loc[sl, ["usdjpy", "us10y", "jgb10y", "spread10y", "us2y", "jgb2y", "spread2y"]].round(3))

In [ ]:
# --- 8. (Colab only, optional) copy processed outputs to Google Drive --------------------------
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    import shutil
    dest = pathlib.Path("/content/drive/MyDrive/FOREX-MODEL/data/processed"); dest.mkdir(parents=True, exist_ok=True)
    for f in (cfg.daily_parquet, cfg.metadata_json):
        shutil.copy(f, dest / f.name)
    print("copied to", dest)